# LLM Systems Lab: Nsight Systems Profiler
This notebook uses your exact code and configs to run the Nsight Systems profiler (`nsys`).

In [ ]:
%uv pip install humanfriendly matplotlib numpy pytest>=8 regex>=2026.3.32 torch~=2.11.0 tqdm>=4.67.1 wandb>=0.19.7 pandas>=2 ty>=0.0.29 ruff>=0.15.10

In [ ]:
%%writefile profile_notebook.py
import torch
import gc
import timeit
import torch.cuda.nvtx as nvtx
from gpu_systems.model_configs import load_all_configs
from gpu_systems.benchmark import benchmark_model

if __name__ == "__main__":
    configs = load_all_configs()
    config = configs.get("medium", list(configs.values())[0])
    
    print("Warming up CUDA...")
    benchmark_model(configs_dict=config, w=3, n=0, mode="full", device="cuda")
    torch.cuda.empty_cache()
    gc.collect()
    
    print("Profiling Full Step...")
    benchmark_model(configs_dict=config, w=1, n=3, mode="full", device="cuda")
    print("Done profiling.")

In [ ]:
import os
os.environ["LD_PRELOAD"] = "/usr/local/cuda/lib64/libcudart.so"

!/usr/local/cuda/bin/nsys profile \
    --trace=cuda,cudnn,cublas,osrt,nvtx \
    --output=notebook_profile_output \
    --force-overwrite=true \
    python3 profile_notebook.py

In [ ]:
!/usr/local/cuda/bin/nsys stats --report nvtxsum notebook_profile_output.nsys-rep